In [19]:
def find_suspicious_transaction(transactions):
    lenth = len(transactions)
    
    for i in range(lenth - 2):
        trans1, trans2, trans3 = transactions[i], transactions[i+1], transactions[i+2]
        
        if trans1 < trans2 < trans3 and (trans1 + trans2 + trans3) >= 10000:
            return i
    
    return -1

transactions = [5000, 3000, 4000, 6000]
#transactions = [2500, 3000, 4500, 2000, 3000, 5000]
#transactions = [2000, 3000, 6000, 1000]
#transactions = [2000, 3000, 6000, 7000]
find_suspicious_transaction(transactions)

1

In [32]:
# Safe import to avoid issues if built-in names like 'list' are overwritten
import builtins

def process_transactions(accounts, transactions):
    """
    Process banking transactions and return final balances.
    """

    # Validate inputs (use builtins.list to avoid Jupyter override issues)
    if not isinstance(accounts, builtins.list) or not isinstance(transactions, builtins.list):
        return {}

    # Initialize account balances
    account_balance = {}
    for acc in accounts:
        if isinstance(acc, tuple) and len(acc) == 2:
            acc_id, balance = acc
            if isinstance(acc_id, str) and isinstance(balance, (int, float)):
                account_balance[acc_id] = balance

    # Track processed transaction IDs (to avoid duplicates)
    processed_txn_ids = set()

    # Process transactions in order
    for txn in transactions:
        if not (isinstance(txn, tuple) and len(txn) == 4):
            continue  # skip invalid format

        txn_id, acc_id, operation, amount = txn

        # Basic validation
        if not isinstance(txn_id, str) or not isinstance(acc_id, str):
            continue
        if not isinstance(operation, str) or not isinstance(amount, (int, float)):
            continue
        if amount < 0:
            continue

        # Skip duplicate transaction IDs
        if txn_id in processed_txn_ids:
            continue
        processed_txn_ids.add(txn_id)

        # Skip if account does not exist
        if acc_id not in account_balance:
            continue

        # Apply transaction
        if operation.lower() == 'credit':
            account_balance[acc_id] += amount

        elif operation.lower() == 'debit':
            if account_balance[acc_id] >= amount:
                account_balance[acc_id] -= amount
            # else ignore (no negative balance allowed)

        # Ignore invalid operations silently

    return account_balance


# =========================
# TEST CASES
# =========================

print("Test Case 1")
accounts1 = [('A1', 1000), ('A2', 500)]
transactions1 = [
    ('T1', 'A1', 'debit', 300),
    ('T2', 'A1', 'credit', 200),
    ('T2', 'A1', 'credit', 200),  # duplicate
    ('T3', 'A2', 'debit', 600),   # should be ignored
    ('T4', 'A3', 'credit', 100)   # invalid account
]
print(process_transactions(accounts1, transactions1))
print("Expected: {'A1': 900, 'A2': 500}")
print()


print("Test Case 2")
accounts2 = [('A1', 1000), ('A2', 2000)]
transactions2 = [
    ('T1', 'A1', 'debit', 500),
    ('T1', 'A2', 'debit', 1000),  # duplicate txn ID
    ('T2', 'A2', 'credit', 300)
]
print(process_transactions(accounts2, transactions2))
print("Expected: {'A1': 500, 'A2': 2300}")
print()


print("Test Case 3 (Edge Cases)")
accounts3 = [('A1', 0)]
transactions3 = [
    ('T1', 'A1', 'debit', 0),     # valid
    ('T2', 'A1', 'debit', 10),    # invalid (insufficient balance)
    ('T3', 'A1', 'credit', 100),  # valid
    ('T4', 'A1', 'invalid', 50),  # invalid operation
    ('T5', 'A2', 'credit', 100),  # invalid account
]
print(process_transactions(accounts3, transactions3))
print("Expected: {'A1': 100}")

Test Case 1
{'A1': 900, 'A2': 500}
Expected: {'A1': 900, 'A2': 500}

Test Case 2
{'A1': 500, 'A2': 2300}
Expected: {'A1': 500, 'A2': 2300}

Test Case 3 (Edge Cases)
{'A1': 100}
Expected: {'A1': 100}


In [1]:
import numpy as np

def count_passed_students(marks):
    """
    Counts number of students who passed based on:
    - Average >= 50
    - No subject marks < 35

    Args:
        marks: 2D list of integers

    Returns:
        int: number of students passed
    """

    # -----------------------------
    # Input Validation
    # -----------------------------
    
    if not isinstance(marks, list) or len(marks) == 0:
        return 0

    try:
        # Convert to NumPy array
        arr = np.array(marks)
    except Exception:
        return 0

    # Ensure 2D structure
    if arr.ndim != 2:
        return 0

    # Ensure numeric values
    if not np.issubdtype(arr.dtype, np.number):
        return 0

    # -----------------------------
    # Core Logic (Vectorized)
    # -----------------------------

    # Rule 1: Check if any subject < 35 (fail condition)
    no_subject_failed = np.all(arr >= 35, axis=1)

    # Rule 2: Average >= 50
    avg_marks = np.mean(arr, axis=1)
    avg_pass = avg_marks >= 50

    # Final condition: both must be true
    passed_students = no_subject_failed & avg_pass

    # Count number of True values
    return int(np.sum(passed_students))


# =========================
# TEST CASES
# =========================

print("Test Case 1")
marks1 = [
    [50, 50, 50],
    [47, 64, 60],
    [90, 92, 88]
]
print(count_passed_students(marks1))
print("Expected: 3")
print()


print("Test Case 2")
marks2 = [
    [60, 70, 80],
    [40, 90, 100],
    [80, 85, 90]
]
print(count_passed_students(marks2))
print("Expected: 2")
print()


print("Test Case 3 (Fail due to subject < 35)")
marks3 = [
    [80, 90, 100],
    [34, 90, 95],   # fail (34 < 35)
    [60, 60, 60]
]
print(count_passed_students(marks3))
print("Expected: 2")
print()


print("Test Case 4 (Fail due to low average)")
marks4 = [
    [40, 40, 40],   # avg < 50
    [50, 50, 50],
    [70, 80, 90]
]
print(count_passed_students(marks4))
print("Expected: 2")
print()


print("Test Case 5 (Edge case: single student)")
marks5 = [
    [55, 60, 65]
]
print(count_passed_students(marks5))
print("Expected: 1")
print()


print("Test Case 6 (All fail)")
marks6 = [
    [10, 20, 30],
    [30, 30, 30],
    [34, 34, 34]
]
print(count_passed_students(marks6))
print("Expected: 0")

Test Case 1
3
Expected: 3

Test Case 2
3
Expected: 2

Test Case 3 (Fail due to subject < 35)
2
Expected: 2

Test Case 4 (Fail due to low average)
2
Expected: 2

Test Case 5 (Edge case: single student)
1
Expected: 1

Test Case 6 (All fail)
0
Expected: 0


You are working in a manufacturing unit where sensor data is collected for multiple products. Each product is tested across multiple quality parameters, and the readings are stored in a 2D list. Each row represents a product, and each column represents a quality parameter score. A product is considered defective if:

Any parameter value is less than 40, or
The average of all parameter values is less than 60
You are given an implementation that attempts to count the number of defective products using NumPy, but it produces incorrect results. You need to debug and fix the code.

Input Format

A 2D list of integers where each row represents a product and each column represents a quality parameter

Output Format

A single integer representing the number of defective products

Sample Test Case 1 - Input
products = [
    [70, 75, 80],
    [60, 65, 70],
    [30, 85, 90],
    [55, 50, 45]
]
Sample Test Case 1 - Output

2

Sample Test Case 2 - Input

products = [
    [40, 30, 30],
    [39, 30, 30],
    [30, 30, 30],
    [35, 30, 35]
]

Sample Test Case 2 - Output

4
Constraints

There is no missing data
There is data for at least one product

In [2]:
import numpy as np
import builtins  # to avoid Jupyter 'list' override issue

def count_defective_products(products):
    """
    Counts number of defective products based on:
    - Any parameter < 40 OR
    - Average < 60
    """

    # -----------------------------
    # Input Validation
    # -----------------------------
    if not isinstance(products, builtins.list) or len(products) == 0:
        return 0

    try:
        arr = np.array(products)
    except:
        return 0

    # Ensure 2D data
    if arr.ndim != 2:
        return 0

    # -----------------------------
    # Core Logic
    # -----------------------------

    # Condition 1: Any value < 40
    cond_low_value = np.any(arr < 40, axis=1)

    # Condition 2: Average < 60
    cond_low_avg = np.mean(arr, axis=1) < 60

    # Defective if ANY condition is true
    defective = cond_low_value | cond_low_avg

    return int(np.sum(defective))


# =========================
# TEST CASES
# =========================

print("Test Case 1")
products1 = [
    [70, 75, 80],
    [60, 65, 70],
    [30, 85, 90],
    [55, 50, 45]
]
print(count_defective_products(products1))  # Expected: 2
print()


print("Test Case 2")
products2 = [
    [40, 30, 30],
    [39, 30, 30],
    [30, 30, 30],
    [35, 30, 35]
]
print(count_defective_products(products2))  # Expected: 4
print()


print("Test Case 3 (No defects)")
products3 = [
    [70, 80, 90],
    [60, 60, 60]
]
print(count_defective_products(products3))  # Expected: 0
print()


print("Test Case 4 (All defective)")
products4 = [
    [10, 20, 30],
    [50, 50, 50],
    [39, 80, 90]
]
print(count_defective_products(products4))  # Expected: 3
print()


print("Test Case 5 (Single product)")
products5 = [
    [59, 60, 61]  # avg < 60 → defective
]
print(count_defective_products(products5))  # Expected: 1

Test Case 1
2

Test Case 2
4

Test Case 3 (No defects)
0

Test Case 4 (All defective)
3

Test Case 5 (Single product)
0


Question 5
You are working as a data analyst in an e-commerce company where customer reviews are stored in a nested dictionary format, with each product containing its details and a set of user reviews, where each review includes a text ('rev') and a rating ('rating'). Your task is to implement a function worst_product(data, mode) that processes this data to compute, for each product, the average rating and the number of reviews containing negative sentiment keywords such as 'bad', 'worst', or 'poor' (ensuring each review contributes at most once to the negative count), and then returns the product ID based on the mode specified. If the mode is 'ratings', then return the product ID with the lowest average rating. If the mode is 'reviews', then return the product ID with the highest number of negative reviews.

Input Format

A nested dictionary of the form:

{
    pid: {
        "pn": product_name,
        "pd": product_description,
        "pr": {
            user_id: {
                "rev": review_text,
                "rating": rating_value,
            },
        },
    },
}
Output Format

The relevant product ID

Sample Test Case 1 - Input

{
    101: {
        "pn": "Smartphone",
        "pd": "Latest model",
        "pr": {
            "u1": {
                "rev": "good product",
                "rating": 4,
            },
            "u2": {
                "rev": "bad battery",
                "rating": 2,
            },
        },
    },
    102: {
        "pn": "Laptop",
        "pd": "Lightweight laptop",
        "pr": {
            "u3": {
                "rev": "excellent",
                "rating": 5,
            },
            "u4": {
                "rev": "great performance",
                "rating": 4,
            },
        },
    },
    103: {
        "pn": "Headphones",
        "pd": "Noise cancelling",
        "pr": {
            "u5": {
                "rev": "worst sound",
                "rating": 1,
            },
            "u6": {
                "rev": "poor quality",
                "rating": 2,
            },
        },
    },
}
Sample Test Case 1 - Output

103 (in both modes)
Sample Test Case 2 - Input

{
    201: {
        "pn": "Tablet",
        "pd": "10 inch display",
        "pr": {
            "u1": {
                "rev": "average",
                "rating": 3,
            },
            "u2": {
                "rev": "fine but weird",
                "rating": 2,
            },
        },
    },
    202: {
        "pn": "Camera",
        "pd": "DSLR",
        "pr": {
            "u3": {
                "rev": "excellent clarity",
                "rating": 5,
            },
            "u4": {
                "rev": "bad screen",
                "rating": 4,
            },
        },
    },
}
Sample Test Case 2 - Output

201 (in ratings mode)
202 (in reviews mode)
Constraints

Each product has at least one review
Ratings are integers between 1 and 5
Review text is lowercase and contains no punctuation

In [3]:
def worst_product(data, mode):
    """
    Returns product ID based on:
    - 'ratings': lowest average rating
    - 'reviews': highest negative review count
    """

    # Validate input
    if not isinstance(data, dict) or mode not in ("ratings", "reviews"):
        return None

    negative_keywords = {"bad", "worst", "poor"}

    result = {}

    # Process each product
    for pid, details in data.items():

        # Defensive checks
        if not isinstance(details, dict):
            continue

        reviews = details.get("pr", {})

        if not isinstance(reviews, dict) or len(reviews) == 0:
            continue

        total_rating = 0
        count = 0
        negative_count = 0

        # Process each review
        for user, review_data in reviews.items():

            if not isinstance(review_data, dict):
                continue

            text = review_data.get("rev", "")
            rating = review_data.get("rating", None)

            # Validate rating
            if not isinstance(rating, int):
                continue

            total_rating += rating
            count += 1

            # Check negative sentiment (only once per review)
            if isinstance(text, str):
                words = set(text.lower().split())
                if words & negative_keywords:
                    negative_count += 1

        # Avoid division by zero
        avg_rating = total_rating / count if count > 0 else float('inf')

        result[pid] = {
            "avg_rating": avg_rating,
            "negative_count": negative_count
        }

    # -----------------------------
    # Final Selection
    # -----------------------------

    if not result:
        return None

    if mode == "ratings":
        # Product with lowest average rating
        return min(result, key=lambda x: result[x]["avg_rating"])

    elif mode == "reviews":
        # Product with highest negative review count
        return max(result, key=lambda x: result[x]["negative_count"])


# =========================
# TEST CASES
# =========================

print("Test Case 1")

data1 = {
    101: {
        "pn": "Smartphone",
        "pd": "Latest model",
        "pr": {
            "u1": {"rev": "good product", "rating": 4},
            "u2": {"rev": "bad battery", "rating": 2},
        },
    },
    102: {
        "pn": "Laptop",
        "pd": "Lightweight laptop",
        "pr": {
            "u3": {"rev": "excellent", "rating": 5},
            "u4": {"rev": "great performance", "rating": 4},
        },
    },
    103: {
        "pn": "Headphones",
        "pd": "Noise cancelling",
        "pr": {
            "u5": {"rev": "worst sound", "rating": 1},
            "u6": {"rev": "poor quality", "rating": 2},
        },
    },
}

print(worst_product(data1, "ratings"))  # Expected: 103
print(worst_product(data1, "reviews"))  # Expected: 103
print()


print("Test Case 2")

data2 = {
    201: {
        "pn": "Tablet",
        "pd": "10 inch display",
        "pr": {
            "u1": {"rev": "average", "rating": 3},
            "u2": {"rev": "fine but weird", "rating": 2},
        },
    },
    202: {
        "pn": "Camera",
        "pd": "DSLR",
        "pr": {
            "u3": {"rev": "excellent clarity", "rating": 5},
            "u4": {"rev": "bad screen", "rating": 4},
        },
    },
}

print(worst_product(data2, "ratings"))  # Expected: 201
print(worst_product(data2, "reviews"))  # Expected: 202

Test Case 1
103
103

Test Case 2
201
202
